# 🔒 Anti-Fabrication Toolkit V3 (Hardened) - Dynamic Auto-Detect

This notebook automatically scans your Kaggle/Colab environment for any datasets, registers them dynamically with the toolkit, and runs verification. 

**Prerequisites:**
1. Upload `anti_fabrication_toolkit_cloud_ready.zip` as a dataset.
2. Upload any number of datasets containing `images` and `masks` folders.
3. Upload your `src` folder containing `verify_strict.py`.

In [ ]:
import os

base_path = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
input_path = '/kaggle/input' if os.path.exists('/kaggle/input') else '/content'

# Unzip toolkit
toolkit_zip = None
for root, dirs, files in os.walk(input_path):
    for f in files:
        if 'anti_fabrication_toolkit' in f and f.endswith('.zip'):
            toolkit_zip = os.path.join(root, f)
            break

if toolkit_zip:
    print(f'Unpacking toolkit from {toolkit_zip}')
    !unzip -q -o "{toolkit_zip}" -d {base_path}/
else:
    print('WARNING: Toolkit zip not found! Please attach it.')


In [ ]:
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display, HTML

toolkit_py = os.path.join(base_path, 'anti_fabrication_toolkit', 'verifyai.py')
eval_script = os.path.join(base_path, 'src', 'verify_strict.py')
master_pass = b"cloud_pass_123\n"

# 1. Initialize Toolkit
print("Initializing Master Password...")
subprocess.run([sys.executable, toolkit_py, "init"], input=master_pass * 2, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 2. Find Datasets
datasets_found = []
for root, dirs, files in os.walk(input_path):
    if 'images' in dirs and 'masks' in dirs:
        dataset_name = os.path.basename(root)
        datasets_found.append({"name": dataset_name, "path": root})

print(f"Found {len(datasets_found)} datasets: {[d['name'] for d in datasets_found]}")

# 3. Register & Run Each
summary_results = []

for ds in datasets_found:
    print(f"\n{'='*50}\nProcessing {ds['name']}...\n{'='*50}")
    
    # Register
    cmd_add = [
        sys.executable, toolkit_py, "add-project", "chakramodel", ds['name'],
        "--dataset-root", ds['path'],
        "--script", eval_script,
        "--section", ds['name']
    ]
    subprocess.run(cmd_add, input=master_pass, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # Run with Environment Variables to make verify_strict.py dynamic!
    env = os.environ.copy()
    env["VERIFY_DATASET_NAME"] = ds['name']
    env["VERIFY_DATASET_ROOT"] = ds['path']
    
    cmd_run = [sys.executable, toolkit_py, "run", "chakramodel", ds['name']]
    
    process = subprocess.Popen(cmd_run, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    # Capture output live
    output_lines = []
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
        output_lines.append(line)
    
    process.stdout.close()
    process.wait()
    
    # Parse Output for Summary Table
    res = {"Dataset": ds['name']}
    for line in output_lines:
        if "Total Evaluated Images:" in line:
            res["Images"] = int(line.split(":")[1].strip())
        elif "Final True Average" in line:
            parts = line.split("Final True Average")[1].split(":")
            metric_name = parts[0].strip()
            metric_val = float(parts[1].strip())
            res[metric_name] = f"{metric_val:.4f}"
        elif "FINAL VERDICT" in line:
            res["Security Verdict"] = line.split(":")[1].strip()
            
    summary_results.append(res)

print("\n\n" + "*"*60)
print("✅ FINAL BENCHMARK SUMMARY")
print("*"*60)

if summary_results:
    df = pd.DataFrame(summary_results)
    display(HTML(df.to_html(index=False)))
else:
    print("No datasets processed.")
